Imports & Relative Path Setup

In [ ]:
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import json

# --- CONFIGURATION ---
# Raw Download Folder (Input)
SOURCE_ROOT = Path("archive/PNG") 
VALID_EXTS = {'.jpg', '.jpeg', '.png'}

# Cleaned Dataset Folder (Output)
SPLIT_CSV = SOURCE_ROOT / "metadata.csv" 

# Split Ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42

print(f"🔧 Project Root: {Path.cwd()}")
print(f"📄 Split file:   {SPLIT_CSV}")

Verify Data

In [ ]:
def verify_raw_data(source_dir):
    """Checks if raw Kvasir-SEG data is valid before we split it."""
    image_dir = source_dir / "Original"
    mask_dir = source_dir / "Ground Truth"
    
    if not image_dir.exists() or not mask_dir.exists():
        print(f"⚠️ Raw data not found at {source_dir}. Skipping verification.")
        return False

    images = sorted([f.name for f in image_dir.iterdir() if f.name != ".DS_Store"])
    masks = sorted([f.name for f in mask_dir.iterdir() if f.name != ".DS_Store"])
    
    # 1. Count Check
    if len(images) != len(masks):
        print(f"❌ Mismatch: {len(images)} images vs {len(masks)} masks")
        return False
        
    # 2. Spot Check (First 5)
    print(f"🔍 Verifying alignment on first 5 samples...")
    for img_name in images[:5]:
        img_p = str(image_dir / img_name)
        mask_p = str(mask_dir / img_name)
        
        if cv2.imread(img_p).shape[:2] != cv2.imread(mask_p).shape[:2]:
            print(f"❌ Dimension mismatch: {img_name}")
            return False

    print("✅ Raw data looks good!")
    return True

# Run Verification
verify_raw_data(SOURCE_ROOT)

Rotate, blur, brighten, and split data.

In [ ]:
def finalize_and_augment_dataset():
    augmented_root = Path.cwd() / "finalized_augmented_dataset_b"
    split_csv = augmented_root / "metadata.csv"

    # 1. Skip if already done
    if split_csv.exists():
        print(f"✅ Split already frozen at '{split_csv}'. Loading...")
        df = pd.read_csv(split_csv)
        print(df['split'].value_counts().to_string())
        return augmented_root, df

    source_root = SOURCE_ROOT
    img_dir  = source_root / "Original"
    mask_dir = source_root / "Ground Truth"

    if not img_dir.exists() or not mask_dir.exists():
        print(f"❌ Expected subfolders 'Original' and 'Ground Truth' inside {source_root} not found.")
        return None, pd.DataFrame([])

    # 2. Find valid pairs and assign splits
    img_stems  = {p.stem for p in img_dir.iterdir()  if p.suffix.lower() in VALID_EXTS}
    mask_stems = {p.stem for p in mask_dir.iterdir() if p.suffix.lower() in VALID_EXTS}
    paired = sorted(img_stems & mask_stems)

    rng = random.Random(SEED)
    rng.shuffle(paired)

    n       = len(paired)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    tags    = ['train'] * n_train + ['val'] * n_val + ['test'] * (n - n_train - n_val)

    # 3. Create output directories
    aug_img_dir = augmented_root / "Original"
    aug_mask_dir = augmented_root / "Ground Truth"
    aug_img_dir.mkdir(parents=True, exist_ok=True)
    aug_mask_dir.mkdir(parents=True, exist_ok=True)

    final_rows = []
    print(f"🔧 Generating augmented dataset under {augmented_root}...")

    # 4. Process files and augment ONLY the train set
    for stem, split in zip(paired, tags):
        img_file  = next(img_dir.glob(f"{stem}.*"))
        mask_file = next(mask_dir.glob(f"{stem}.*"))

        img = cv2.imread(str(img_file))
        mask = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            continue

        # ALWAYS save the original image/mask pair (Train, Val, & Test)
        out_img = aug_img_dir / img_file.name
        out_mask = aug_mask_dir / mask_file.name
        cv2.imwrite(str(out_img), img)
        cv2.imwrite(str(out_mask), mask)

        final_rows.append({
            'stem': stem,
            'image_path': f"Original/{out_img.name}",
            'mask_path': f"Ground Truth/{out_mask.name}",
            'split': split
        })

        # ONLY augment if this image belongs to the train split
        if split == 'train':
            # Rotation
            angle = random.uniform(0, 360)
            h, w = img.shape[:2]
            center = (w / 2.0, h / 2.0)
            rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)

            aug_img = cv2.warpAffine(img, rot_mat, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
            aug_mask = cv2.warpAffine(mask, rot_mat, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT)

            stem_rot = f"{stem}_r{int(angle)}"
            out_img_rot = aug_img_dir / f"{stem_rot}{img_file.suffix}"
            out_mask_rot = aug_mask_dir / f"{stem_rot}{mask_file.suffix}"
            cv2.imwrite(str(out_img_rot), aug_img)
            cv2.imwrite(str(out_mask_rot), aug_mask)

            final_rows.append({
                'stem': stem_rot,
                'image_path': f"Original/{out_img_rot.name}",
                'mask_path': f"Ground Truth/{out_mask_rot.name}",
                'split': 'train'
            })

            # Brightness
            bright = np.clip(img.astype(np.float32) * 1.3, 0, 255).astype(np.uint8)
            stem_bright = f"{stem}_bright"
            out_img_bright = aug_img_dir / f"{stem_bright}{img_file.suffix}"
            out_mask_bright = aug_mask_dir / f"{stem_bright}{mask_file.suffix}"
            cv2.imwrite(str(out_img_bright), bright)
            cv2.imwrite(str(out_mask_bright), mask)

            final_rows.append({
                'stem': stem_bright,
                'image_path': f"Original/{out_img_bright.name}",
                'mask_path': f"Ground Truth/{out_mask_bright.name}",
                'split': 'train'
            })

            # Blur
            blur = cv2.GaussianBlur(img, (9, 9), 2)
            stem_blur = f"{stem}_blur"
            out_img_blur = aug_img_dir / f"{stem_blur}{img_file.suffix}"
            out_mask_blur = aug_mask_dir / f"{stem_blur}{mask_file.suffix}"
            cv2.imwrite(str(out_img_blur), blur)
            cv2.imwrite(str(out_mask_blur), mask)

            final_rows.append({
                'stem': stem_blur,
                'image_path': f"Original/{out_img_blur.name}",
                'mask_path': f"Ground Truth/{out_mask_blur.name}",
                'split': 'train'
            })

    print("✅ Augmentation complete")
    final_df = pd.DataFrame(final_rows)
    final_df.to_csv(split_csv, index=False)
    print(f"💾 Final augmented split frozen → {split_csv}")
    print(final_df['split'].value_counts().to_string())

    return augmented_root, final_df

# Run the execution and update our global variables
SOURCE_ROOT, splits_df = finalize_and_augment_dataset()
SPLIT_CSV = SOURCE_ROOT / "metadata.csv"

Augment Samples

Show augmented samples

In [ ]:
def show_extreme_augmented_samples(source_root):
    image_dir = Path(source_root) / "Original"
    bright_candidates = []
    blur_candidates = []

    for img_path in sorted(image_dir.iterdir()):
        if img_path.suffix.lower() not in VALID_EXTS:
            continue
        stem = img_path.stem
        if "_bright" in stem or "_blur" in stem:
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            brightness = float(np.mean(cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 2]))
            blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())

            if "_bright" in stem:
                bright_candidates.append((brightness, img_path, img))
            elif "_blur" in stem:
                blur_candidates.append((blur_score, img_path, img))

    if not bright_candidates:
        print("⚠️ No bright augmented samples found.")
        return
    if not blur_candidates:
        print("⚠️ No blur augmented samples found.")
        return

    brightest_path = max(bright_candidates, key=lambda item: item[0])[1]
    blurriest_path = min(blur_candidates, key=lambda item: item[0])[1]
    brightest_img = cv2.cvtColor(cv2.imread(str(brightest_path)), cv2.COLOR_BGR2RGB)
    blurriest_img = cv2.cvtColor(cv2.imread(str(blurriest_path)), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(brightest_img)
    axes[0].set_title(f"Maximum Brightness\n{brightest_path.name}")
    axes[0].axis("off")

    axes[1].imshow(blurriest_img)
    axes[1].set_title(f"Maximum Blur\n{blurriest_path.name}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

show_extreme_augmented_samples(SOURCE_ROOT)

Data Set Creation

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
%pip install torch torchvision
from torch.utils.data import Dataset, DataLoader # Assuming PyTorch usage
from PIL import Image

# --- RELATIVE PATH SETUP ---
# Get the directory where this notebook is running
BASE_DIR = Path.cwd()

# Define the path to your data folder relative to this notebook
DATA_DIR = BASE_DIR / "Kvasir-SEG"

print(f"✅ Project Root: {BASE_DIR}")
print(f"📂 Data Directory: {DATA_DIR}")

# Check if it exists
if not DATA_DIR.exists():
    print("❌ Error: 'Kvasir-SEG' folder not found. Did you run the split script?")
else:
    print("Found 'data' folder. Ready to load.")

Dataset Class

In [ ]:
class KvasirDataset(Dataset):
    def __init__(self, split="train", transform=None):
        self.root  = SOURCE_ROOT
        self.split = split

        if not SPLIT_CSV.exists():
            raise FileNotFoundError("metadata.csv not found — run create_split() first.")

        df  = pd.read_csv(SPLIT_CSV)
        sub = df[df['split'] == split].reset_index(drop=True)

        self.image_paths = [self.root / p for p in sub['image_path']]
        self.mask_paths  = [self.root / p for p in sub['mask_path']]
        self.stems       = sub['stem'].tolist()

    def __len__(self):
        return len(self.image_paths)   # ← fixed

    def __getitem__(self, idx):
        img_path  = str(self.image_paths[idx])
        mask_path = str(self.mask_paths[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        _, mask = cv2.threshold(mask, 127, 1.0, cv2.THRESH_BINARY)
        mask = mask.astype(np.float32)

        return image, mask

    def raw(self, idx):
        image = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        return image, mask

Confirm Split is Frozen

In [ ]:
def check_split_frozen():
    df    = pd.read_csv(SPLIT_CSV)
    train = set(df[df.split == 'train'].stem)
    val   = set(df[df.split == 'val'].stem)
    test  = set(df[df.split == 'test'].stem)

    checks = {'train∩val': train & val, 'train∩test': train & test, 'val∩test': val & test}
    all_ok = True
    for name, overlap in checks.items():
        if overlap:
            print(f"❌ Overlap in {name}: {len(overlap)} items → {list(overlap)[:3]}")
            all_ok = False
        else:
            print(f"✅ No overlap: {name}")

    total = len(df)
    print(f"\nCounts:    {df.split.value_counts().to_dict()}")
    print(f"Fractions: train={len(train)/total:.1%}  val={len(val)/total:.1%}  test={len(test)/total:.1%}")
    print(f"\n{'✅ Split is clean and frozen.' if all_ok else '❌ Fix overlaps before training!'}")

check_split_frozen()

Mask alignment and Resolution Consistency

In [ ]:
def check_resolution_consistency(sample_n=0):
    """
    Checks every (or sample_n random) pairs for:
      - image and mask having the same H×W
      - mask pixels being only {0, 255}
    """
    df = pd.read_csv(SPLIT_CSV)
    if sample_n:
        df = df.sample(min(sample_n, len(df)), random_state=SEED)

    mismatched, bad_vals, res_counts = [], [], {}

    for _, row in df.iterrows():
        img  = cv2.imread(str(SOURCE_ROOT / row.image_path))
        mask = cv2.imread(str(SOURCE_ROOT / row.mask_path), cv2.IMREAD_GRAYSCALE)

        ih, iw = img.shape[:2]
        mh, mw = mask.shape[:2]
        res_counts[(iw, ih)] = res_counts.get((iw, ih), 0) + 1

        if (ih, iw) != (mh, mw):
            mismatched.append((row.stem, (iw, ih), (mw, mh)))

        unique = set(mask.flatten().tolist())
        if not unique.issubset({0, 255}):
            bad_vals.append((row.stem, unique))

    print(f"Checked {len(df)} pairs\n")

    if mismatched:
        print(f"❌ {len(mismatched)} resolution mismatches:")
        for s, i, m in mismatched[:5]:
            print(f"   {s}: image={i}  mask={m}")
    else:
        print("✅ All image/mask pairs share the same resolution.")

    if bad_vals:
        print(f"⚠️  {len(bad_vals)} masks with unexpected pixel values (not {{0,255}}):")
        for s, v in bad_vals[:5]:
            print(f"   {s}: {v}")
    else:
        print("✅ All mask pixels are {0, 255}.")

    print(f"\nTop-5 resolutions (W×H):")
    for (w, h), cnt in sorted(res_counts.items(), key=lambda x: -x[1])[:5]:
        print(f"  {w}×{h} → {cnt} images")

check_resolution_consistency(sample_n=0)  # 0 = check all 1000

Overlay Sanity Check

In [ ]:
train_ds = KvasirDataset(split="train")
val_ds   = KvasirDataset(split="val")
test_ds  = KvasirDataset(split="test")

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


Sanity Check & Visualization

In [ ]:
def overlay_sanity_check(dataset, n_samples=6, seed=42):
    """
    Plots image | mask | green overlay for random samples.
    If green doesn't sit on the polyp → alignment is broken.
    """
    rng  = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), min(n_samples, len(dataset)))

    fig, axes = plt.subplots(n_samples, 3, figsize=(13, 4.2 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for row, idx in enumerate(idxs):
        image, mask = dataset.raw(idx)
        stem        = dataset.stems[idx]

        # Resize mask to image size if they differ (shouldn't happen after check above)
        if image.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]),
                              interpolation=cv2.INTER_NEAREST)

        # Build green overlay
        overlay = image.copy().astype(float)
        poly    = mask == 255
        overlay[poly, 0]  = overlay[poly, 0] * 0.4
        overlay[poly, 1]  = np.clip(overlay[poly, 1] * 0.4 + 180, 0, 255)
        overlay[poly, 2]  = overlay[poly, 2] * 0.4
        overlay = overlay.astype(np.uint8)

        h, w       = image.shape[:2]

        axes[row][0].imshow(image)
        axes[row][0].set_title(f"{stem}\n{w}×{h}", fontsize=8)
        axes[row][0].axis('off')

        axes[row][1].imshow(mask, cmap='gray')
        axes[row][1].set_title(f"Mask  |  polyp")
        axes[row][1].axis('off')

        axes[row][2].imshow(overlay)
        axes[row][2].set_title("Overlay (green = polyp)", fontsize=8)
        axes[row][2].axis('off')

    plt.suptitle(f"Mask Alignment — {dataset.split} split", fontsize=13,
                 fontweight='bold', y=1.005)
    plt.tight_layout()
    out = SOURCE_ROOT / f"overlay_check_{dataset.split}.png"
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved → {out}")

overlay_sanity_check(train_ds, n_samples=6, seed=SEED)
overlay_sanity_check(val_ds,   n_samples=4, seed=SEED)
overlay_sanity_check(test_ds,  n_samples=4, seed=SEED)